# Always Plot Your Data
### An interactive guide to data visualization

**How to use this notebook:**  
Work through it top to bottom. Run each code cell. When you see a **Check your understanding** box, type your answer and click *Check*.  
There are no trick questions — but some answers might surprise you.

---
*This notebook takes about 2–3 hours to complete thoughtfully.  
It is designed for the self-study day (12 May). Take your time.*

In [ ]:
pip install ipywidgets

In [3]:
# Run this cell first — it loads all libraries and sets up the quiz widgets.

import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

%matplotlib inline
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

_QUIZ_DIR = Path("quiz_data")


# ── Quiz helpers ──────────────────────────────────────────────────────────────

def make_mc_quiz(question, options, correct_idx, explanations):
    q = widgets.HTML(
        f'<div style="font-size:1.05em;font-weight:600;color:#dcd7ba;margin-bottom:8px;">'
        f'🤔 {question}</div>'
    )
    radio = widgets.RadioButtons(
        options=options,
        layout=widgets.Layout(margin='4px 0')
    )
    btn = widgets.Button(
        description='Check',
        button_style='primary',
        layout=widgets.Layout(width='80px', margin='8px 0')
    )
    out = widgets.Output()

    def on_check(_):
        idx = options.index(radio.value)
        with out:
            clear_output(wait=True)
            if idx == correct_idx:
                display(HTML(
                    f'<span style="color:#98bb6c;font-weight:600">✓ Correct!</span>'
                    f'<div style="color:#c8c093;margin:6px 0 0 16px">{explanations[idx]}</div>'
                ))
            else:
                display(HTML(
                    f'<span style="color:#c34043;font-weight:600">✗ Not quite.</span>'
                    f'<div style="color:#c8c093;margin:6px 0 0 16px">{explanations[idx]}</div>'
                ))

    btn.on_click(on_check)
    display(widgets.VBox(
        [q, radio, btn, out],
        layout=widgets.Layout(
            border='1px solid #2a2a37', padding='14px', margin='10px 0',
            border_radius='6px'
        )
    ))


def run_quiz(name):
    """Load and render a quiz from quiz_data/<name>.json."""
    with open(_QUIZ_DIR / f"{name}.json") as f:
        data = json.load(f)
    make_mc_quiz(
        data["question"], data["options"],
        data["correct_idx"], data["explanations"]
    )


print("✓ Setup complete. All quiz widgets are ready.")

✓ Setup complete. All quiz widgets are ready.


---
## Section 1 — The Trap

In 1973, statistician Francis Anscombe published a short paper in *The American Statistician*:  
**"Graphs in Statistical Analysis"** (Vol. 27, No. 1, pp. 17–21).

His argument was methodological. There was a widespread assumption at the time that numerical calculations are exact while graphs are rough. Anscombe disagreed — and built four datasets to make that argument impossible to dismiss.

Let's start where he started: with the numbers.

In [4]:
# Anscombe's quartet is built into seaborn — no download needed.
anscombe = sns.load_dataset("anscombe")

print(f"Shape: {anscombe.shape}")
print(f"Columns: {list(anscombe.columns)}")
print(f"Datasets: {sorted(anscombe['dataset'].unique())}")
anscombe.head(8)

Shape: (44, 3)
Columns: ['dataset', 'x', 'y']
Datasets: ['I', 'II', 'III', 'IV']


,dataset,x,y
0,I,10.0,8.04
1,I,8.0,6.95
2,I,13.0,7.58
3,I,9.0,8.81
4,I,11.0,8.33
5,I,14.0,9.96
6,I,6.0,7.24
7,I,4.0,4.26


Four datasets — I, II, III, IV — each with eleven (x, y) pairs.  
Anscombe constructed them by hand. The question is: are they similar?

Let's look at the summary statistics.

In [ ]:
# Summary statistics for each dataset.
stats = (
    anscombe
    .groupby("dataset")[["x", "y"]]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)
stats

Look at the `x` columns: mean = 9.0, std = 3.32 for every dataset.  
The `y` columns are also nearly identical: mean ≈ 7.5, std ≈ 2.03.

What about correlation? That measures how strongly x and y move together.

In [ ]:
# Pearson correlation between x and y, for each dataset.
corr = (
    anscombe
    .groupby("dataset")
    .apply(lambda g: g["x"].corr(g["y"]))
    .round(3)
    .rename("Pearson r")
)
print(corr.to_string())

All four correlations are approximately 0.816 — a moderately strong positive relationship.  
And if you fit a linear regression, the slopes and intercepts are nearly identical too.

**By every standard summary statistic, these four datasets are the same.**

In [ ]:
run_quiz("q1")

---
## Section 2 — The Reveal

Now let's plot them.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, (name, group) in zip(axes.flatten(), anscombe.groupby("dataset")):
    ax.scatter(group["x"], group["y"], color="#7e9cd8", s=60, alpha=0.85, zorder=3)

    # Linear regression line
    m, b = np.polyfit(group["x"], group["y"], 1)
    x_range = np.linspace(group["x"].min() - 0.5, group["x"].max() + 0.5, 100)
    ax.plot(x_range, m * x_range + b, color="#c34043", linewidth=1.5, alpha=0.75, zorder=2)

    ax.set_title(f"Dataset {name}  (r = {group['x'].corr(group['y']):.3f})",
                 fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)

fig.suptitle("Anscombe's Quartet — identical statistics, different data",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

Same regression line (red). Same correlation. Completely different shapes.

- **Dataset I** — a linear relationship. The regression line fits well.
- **Dataset II** — clearly non-linear (a curve), yet the correlation is 0.816. The linear model is wrong.
- **Dataset III** — almost perfectly linear, but one outlier drags the regression line off.
- **Dataset IV** — all x-values are the same (x = 8) except one outlier at x = 19. The correlation is driven entirely by that single point.

In three of the four cases, reporting the correlation alone would mislead you.

In [ ]:
run_quiz("q2")

In [ ]:
run_quiz("q3")

---
## Section 3 — Going Deeper: The Datasaurus

In 2016, Alberto Cairo — data journalist and professor at the University of Miami — published a dataset shaped like a dinosaur. He called it the Datasaurus. His point: identical summary statistics can hide dramatically different structures.

In 2017, Justin Matejka and George Fitzmaurice at Autodesk published a paper at the ACM CHI conference.  
They took Cairo's dinosaur as the seed dataset and used *simulated annealing* — an optimization algorithm — to morph it into twelve additional shapes, all while preserving the same mean, standard deviation, and correlation as the original dinosaur.

The result is 13 datasets that share these statistics:

| x̄ | ȳ | σx | σy | r |
|---|---|---|---|---|
| ≈ 54.26 | ≈ 47.83 | ≈ 16.76 | ≈ 26.93 | ≈ −0.06 |

Note: these are *different* from Anscombe's statistics (x̄ = 9, r = 0.816). The algorithm preserves *internal* consistency — it keeps the dinosaur's own statistics across all 13 shapes.

One of those shapes is still a dinosaur.

Let's load all 13. **This cell requires an internet connection.**

In [ ]:
import urllib.request

DATASAURUS_URL = (
    "https://raw.githubusercontent.com/"
    "jumpingrivers/datasauRus/main/inst/extdata/DatasaurusDozen-Long.tsv"
)

try:
    datasaurus = pd.read_csv(DATASAURUS_URL, sep="\t")
    print(f"Loaded successfully. Shape: {datasaurus.shape}")
    print(f"Datasets: {sorted(datasaurus['dataset'].unique())}")
except Exception as e:
    print(f"Could not load from URL: {e}")
    print("Check your internet connection and try again.")
    datasaurus = None

In [ ]:
if datasaurus is not None:
    # Summary statistics for all 13 datasets.
    ds_stats = (
        datasaurus
        .groupby("dataset")[["x", "y"]]
        .agg(["mean", "std"])
        .round(2)
    )
    ds_stats

13 datasets. Every single one has x̄ ≈ 54.3, ȳ ≈ 47.8, σx ≈ 16.8, σy ≈ 26.9.

Before you run the next cell — try to answer this:

In [ ]:
run_quiz("q4")

In [ ]:
if datasaurus is not None:
    dataset_names = sorted(datasaurus["dataset"].unique())
    n = len(dataset_names)
    cols = 4
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.2))
    axes_flat = axes.flatten()

    for i, name in enumerate(dataset_names):
        ax = axes_flat[i]
        g = datasaurus[datasaurus["dataset"] == name]
        ax.scatter(g["x"], g["y"], s=8, alpha=0.7, color="#7e9cd8")
        ax.set_title(name, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(
        "The Datasaurus Dozen — 13 datasets, identical statistics",
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()

One of those shapes is a dinosaur.

All thirteen share the same mean, standard deviation, and Pearson correlation.  
No summary statistic distinguishes them.  
Only a plot can.

> **The rule:** compute summary statistics if you want. Then plot your data. In that order, always.

---

## Section 4 — Choosing the Right Chart

Anscombe taught us to plot. But which plot?

The answer starts with a question — not about the data, but about the viewer:  
**What do I want the viewer to compare?**

| You want the viewer to… | Use |
|---|---|
| Compare values across groups | Bar chart |
| See the distribution of one variable | Histogram, box plot, violin |
| See the relationship between two variables | Scatter plot |
| Track change over time | Line chart |
| See correlation across many variables | Heatmap, pairplot |

The chart type is not an aesthetic choice. It is a *communication decision*.  
Below are three worked examples.

In [ ]:
# Example 1: temperature over time → the viewer should track change.
# Question: "How did the average temperature in Malmö change across the week?"

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
temp_malmo = [6.2, 7.8, 9.1, 11.4, 10.2, 8.5, 7.3]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(days, temp_malmo, marker="o", color="#7fb4ca", linewidth=2)
ax.set_title("Malmö peaked mid-week, then cooled over the weekend")
ax.set_xlabel("Day of week")
ax.set_ylabel("Temperature (°C)")
plt.tight_layout()
plt.show()

Notice: the title is a **claim**, not a label.  
"Temperature by day" is a label. "Malmö peaked mid-week" is a claim — the viewer takes something away.

A line chart is right here because x is ordered time. Connecting points implies continuity.

In [ ]:
run_quiz("q5")

In [ ]:
# Example 2: precipitation vs temperature → the viewer should see a relationship.
# Question: "Is there a relationship between temperature and annual precipitation?"

cities = pd.DataFrame({
    "city":       ["Kiruna", "Umeå",  "Östersund", "Stockholm", "Göteborg", "Malmö"],
    "temp_c":     [-0.5,      3.1,      2.8,          7.4,         9.1,        8.3],
    "precip_mm":  [400,       520,      550,          560,         760,        620],
})

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(cities["temp_c"], cities["precip_mm"],
           s=90, color="#98bb6c", alpha=0.85, zorder=3)

for _, row in cities.iterrows():
    ax.annotate(row["city"],
                xy=(row["temp_c"], row["precip_mm"]),
                xytext=(4, 4), textcoords="offset points",
                fontsize=8, color="#c8c093")

ax.set_title("Warmer Swedish cities tend to receive more precipitation")
ax.set_xlabel("Mean annual temperature (°C)")
ax.set_ylabel("Annual precipitation (mm)")
plt.tight_layout()
plt.show()

Scatter plots are for two numeric variables where neither is a time axis.  
Each point is one observation (one city). The pattern of points reveals the relationship.

Notice the annotation — `ax.annotate()` labels each point directly.  
The viewer doesn't have to match colors to a legend. That's the principle: minimize the distance between mark and meaning.

In [ ]:
run_quiz("q6")

In [ ]:
# Example 3: distribution of values → the viewer should see spread and shape.
# Question: "How are film ratings distributed on this platform?"

rng = np.random.default_rng(42)
# Simulate ratings: most films cluster around 6–7, with a tail toward low scores.
ratings = np.clip(
    np.concatenate([rng.normal(6.8, 1.2, 800), rng.normal(4.0, 1.5, 200)]),
    1, 10
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ratings, bins=30, color="#957fb8", edgecolor="#1f1f28", alpha=0.85)
ax.axvline(np.median(ratings), color="#dca561", linewidth=2,
           label=f"Median = {np.median(ratings):.1f}")
ax.axvline(np.mean(ratings), color="#c34043", linewidth=2, linestyle="--",
           label=f"Mean = {np.mean(ratings):.1f}")
ax.set_title("Most films rate 6–7, but a cluster of poor films pulls the mean down")
ax.set_xlabel("Rating (1–10)")
ax.set_ylabel("Number of films")
ax.legend()
plt.tight_layout()
plt.show()

The histogram reveals the shape of the distribution — something `df.describe()` partially hides.  
Here: two clusters. The mean is pulled toward the lower cluster; the median is more representative.

The vertical lines — `ax.axvline()` — annotate the plot directly. The title states the conclusion.

In [ ]:
run_quiz("q7")

---
## Section 5 — The figure/axes Architecture

You have written `fig, ax = plt.subplots()` many times by now.  
Let's understand what it actually returns — and why it matters.

In [ ]:
fig, ax = plt.subplots()
plt.close()  # don't display an empty plot

print(f"type(fig): {type(fig)}")
print(f"type(ax):  {type(ax)}")

`Figure` is the canvas — the entire image, including margins and whitespace.  
`Axes` is one coordinate system — the area where data is drawn.

A Figure can contain multiple Axes. That is what `plt.subplots(rows, cols)` does.

In [ ]:
# plt.subplots(2, 3) returns a 2-D array of Axes.
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
plt.close()

print(f"axes.shape:  {axes.shape}")
print(f"axes.size:   {axes.size}   (total number of Axes objects)")
print(f"axes[0, 0]:  {type(axes[0, 0])}")   # top-left
print(f"axes[1, 2]:  {type(axes[1, 2])}")   # bottom-right

`axes` is a NumPy array — you already know how to index those (from v19-dag-1).  
`axes[0, 0]` is the top-left. `axes[1, 2]` is the bottom-right.

When you write `for ax in axes.flatten()`, you iterate over all six Axes in order — left to right, top to bottom.

In [ ]:
run_quiz("q8")

---
## Section 6 — The Detective's Method

You know the tools. You know which chart serves which question.  
But when you sit down with a brand-new dataset — one you have never seen — where do you start?

John Tukey developed exploratory data analysis through the 1960s and 70s, publishing the canonical book *Exploratory Data Analysis* in 1977. In its opening paragraph he wrote:

> *"Exploratory data analysis is detective work — numerical detective work or counting detective work — or graphical detective work."*

Detective work. You arrive at a scene without a theory. Your first job is to look carefully before reaching for any explanation.

The most common mistake is skipping this and jumping straight to a scatter plot. That is like running a forensic test before reading the incident report. You are computing statistics about a reality you have not yet grasped.

What works instead is a spiral — each step generates questions that drive the next:

1. **Understand the context** — before you open the file: what is a row? who collected this? when? why?
2. **Inspect mechanically** — shape, data types, missing values, unexpected values
3. **Explore univariately** — one variable at a time; look for anomalies
4. **Explore bivariately** — pairs of variables; anomalies become questions
5. **Ask what is not here** — every dataset has limits on what it can answer

The goal of this process is not answers. It is *good questions* — questions worth visualizing, questions worth testing.

To show this in practice, we will work through a dataset we have never seen before: the **Palmer Penguins** dataset. It contains physical measurements of 344 penguins from three species, collected on three islands in Antarctica between 2007 and 2009.

We know nothing else about it yet. Let's run the spiral.

In [ ]:
penguins = sns.load_dataset("penguins")
print(f"Shape: {penguins.shape}  →  {penguins.shape[0]} rows, {penguins.shape[1]} columns")
penguins.head()

### Step 1 — Context (before code)

Before running any analysis, answer three questions about the data's origin. These answers are not in the DataFrame — but they constrain every conclusion you can draw.

- **What does one row represent?** One penguin, measured once.
- **How was the data collected?** Scientists measured physical features in the field on three Antarctic islands.
- **What population does it cover?** Three species, three islands, three years. Not all penguins. These penguins.

### Step 2 — Mechanical inspection

The mechanical inspection is not about finding insights yet. It is about understanding the raw material — what is actually in each column, what is missing, and whether the ranges make sense.

In [ ]:
print("── Data types ──────────────────────────────────────────")
print(penguins.dtypes)

print("\n── Missing values ──────────────────────────────────────")
missing = penguins.isnull().sum()
print(missing[missing > 0].to_string())

print("\n── Categorical value counts ────────────────────────────")
for col in ["species", "island", "sex"]:
    print(f"\n{col}:")
    print(penguins[col].value_counts(dropna=False).to_string())

print("\n── Numeric summary ─────────────────────────────────────")
penguins.describe().round(1)

Four things to notice — and four questions they generate.

**`sex` has 11 missing values.** These are not random. In the field, some penguins' sex could not be determined from physical measurements alone. Dropping them silently — which `dropna()` would do — biases any analysis of sex differences. That is not a cleaning decision; it is a scientific one.

**The four numeric columns each have exactly 2 missing values.** The same two penguins are almost certainly incomplete across all measurements. Investigate before dropping.

**`body_mass_g` ranges from 2,700 to 6,300 g** — more than a 2× spread. That is too large to be measurement noise. It likely reflects real biological structure: different species, different sexes, or both.

**Three species, three islands.** Are all three species found on all three islands? That question is not answered by `value_counts()` alone — it requires a crosstab.

We have not plotted anything. We already have four questions.

### Step 3 — Univariate exploration

Look at each numeric variable in isolation. The goal here is to find anomalies — shapes that do not match what you would expect from a single, homogeneous population.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

for ax, col in zip(axes, numeric_cols):
    ax.hist(penguins[col].dropna(), bins=22, color="#7e9cd8",
            edgecolor="#1f1f28", alpha=0.9)
    label = col.replace("_mm", "\n(mm)").replace("_g", "\n(g)").replace("_", " ")
    ax.set_title(label, fontsize=9)
    ax.set_ylabel("Count")

plt.suptitle("Palmer Penguins — four numeric variables, univariate", fontsize=10, y=1.03)
plt.tight_layout()
plt.show()

Every one of those distributions is **bimodal** — two humps.

Bimodality in a dataset like this almost always means one thing: the data is a mixture of distinct groups whose distributions overlap. The histogram cannot tell you which group is responsible. A unimodal histogram might mean "one homogeneous population." A bimodal histogram means "this is at least two things pretending to be one."

The most obvious candidate here is species. We know there are three of them. Let's split the scatter by species and see whether the bimodal pattern resolves.

### Step 4 — Bivariate exploration

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: pooled — structure is invisible
ax1.scatter(penguins["bill_length_mm"], penguins["flipper_length_mm"],
            color="#717c7c", alpha=0.45, s=30)
ax1.set_title("Pooled — structure hidden")
ax1.set_xlabel("Bill length (mm)")
ax1.set_ylabel("Flipper length (mm)")

# Right: colored by species — structure becomes visible
colors = {"Adelie": "#7e9cd8", "Gentoo": "#98bb6c", "Chinstrap": "#dca561"}
for species, group in penguins.dropna(subset=["bill_length_mm"]).groupby("species"):
    ax2.scatter(group["bill_length_mm"], group["flipper_length_mm"],
                color=colors[species], alpha=0.75, s=40, label=species)
ax2.set_title("Split by species — structure visible")
ax2.set_xlabel("Bill length (mm)")
ax2.set_ylabel("Flipper length (mm)")
ax2.legend()

plt.suptitle("Same data — two perspectives", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

What looked like noise was structure we had not yet seen.

This is the core dynamic of EDA: an anomaly is not a problem to be cleaned away. It is a question waiting to be asked. The bimodal histogram *was* the question. The colored scatter *was* the answer — and immediately generates the next question: "do all three species separate this clearly on *every* measurement, or only on some?"

That is the spiral. Each answer opens the next question.

### Step 5 — What this data cannot tell us

Before finishing any exploratory session, ask one more question: **what is not here?**

This dataset covers three islands, three species, three years. It cannot speak to other locations, other populations, or other decades. The 11 missing sex values are not random — they represent penguins whose sex could not be determined. Any conclusion about sex differences silently excludes them unless you account for it.

The unit of observation is one penguin, measured once. It cannot tell you how individual penguins change over their lifetimes.

This is not a critique of the dataset. Every dataset has a scope. Knowing the scope is what separates a conclusion from an overreach.

---

**The five steps — compact reference:**

| Step | What you do | Why |
|---|---|---|
| 1. Context | Ask: what is a row? how was this collected? | Answers not in the data constrain every conclusion |
| 2. Mechanical inspection | shape, dtypes, nulls, value ranges | Find the unexpected before you build on it |
| 3. Univariate | histograms, value counts | Anomalies are questions waiting to be asked |
| 4. Bivariate | scatter, box plots by group | Anomalies resolve into structure when you split correctly |
| 5. Epistemological boundary | ask what is absent | A conclusion is only as strong as the data it rests on |

The goal is not answers. It is good questions. Then you make the charts.

---
## Section 7 — Your Turn

You explored a dataset this morning. Now apply what you have learned.

**Before you write a single line of code**, answer this:  
What do I want to show? What comparison do I want the viewer to make?

> **Note:** KK1 will be released later today. When you receive it, you will apply exactly these same steps to that dataset. For now, use the dataset you worked with this morning.

---
### Step 1 — Load your dataset

In [ ]:
# Replace 'your_file.csv' with the path to your dataset.
# Adjust read_csv() parameters as needed (sep, encoding, etc.)

# df = pd.read_csv('your_file.csv')

# Once loaded, run describe() and look at the first few rows.
# df.describe()

# Write one sentence here (as a comment) about something you notice:
# → 

### Step 2 — Visualization 1: Distribution

Choose one numeric column. What does its distribution look like?  
Use a histogram or box plot. Write one sentence below the plot.

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 4))
#
# ax.hist(df['column_name'], bins=30, color='#7fb4ca', edgecolor='#1f1f28')
# ax.set_title('Write a title that makes a claim here')
# ax.set_xlabel('Column name (unit)')
# ax.set_ylabel('Count')
#
# plt.tight_layout()
# plt.show()

# What does this distribution tell you?
# → 

### Step 3 — Visualization 2: Relationship

Choose two numeric columns. Is there a relationship between them?  
Use a scatter plot. Annotate anything surprising.

In [ ]:
# fig, ax = plt.subplots(figsize=(7, 5))
#
# ax.scatter(df['column_x'], df['column_y'], alpha=0.6, s=40)
# ax.set_title('Write a claim here')
# ax.set_xlabel('Column x (unit)')
# ax.set_ylabel('Column y (unit)')
#
# plt.tight_layout()
# plt.show()

# Does the scatter match what you expected from the correlation alone?
# → 

### Step 4 — Visualization 3: Comparison across groups

Choose one categorical column and one numeric column.  
Compare the numeric variable across the categories.  
A bar chart (means), box plot (distributions), or violin plot all work — choose based on what you want to show.

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 5))
#
# sns.boxplot(data=df, x='category_column', y='numeric_column', ax=ax)
# ax.set_title('Write a claim here')
# ax.set_xlabel('Category')
# ax.set_ylabel('Numeric column (unit)')
#
# plt.tight_layout()
# plt.show()

# What is the most notable difference between groups?
# → 

---
### When you are done

Save your notebook. Then commit:

```bash
git add notebooks/v20-dag-1/intro.ipynb
git commit -m "vis: three charts, exploratory"
git push
```

When KK1 is released later today, open a new notebook (or a new section here) and repeat Steps 2–4 for that dataset.  
The charts you produce will be part of your KK1 submission.

---

> *"The greatest value of a picture is when it forces us to notice what we never expected to see."*  
> — John W. Tukey, *Exploratory Data Analysis*, 1977